In [1]:
# BPDRR create breaks
# Mario Castro-Gama
# 2026-05-20

In [3]:
import wntr
import numpy as np
import pandas as pd

In [4]:
# this is to remove warnings from showing as pink blocks as results of CELLS
import warnings
warnings.filterwarnings("ignore")

In [97]:
def split_pipes_with_emitter(input_inp,list_pipes,output_inp):
    """
    Split selected EPANET pipes by inserting a midpoint junction
    with an emitter.

    Modifications:
      - Original pipe is retained but set to CLOSED.
      - New midpoint junction H_<pipe_id> is added.
      - Midpoint junction receives an emitter coefficient.
      - Two new pipes are created with half the original length.

    Parameters
    ----------
    input_inp : str
        Original EPANET INP file.

    list_pipes : list[str]
        Pipe IDs to split.

    output_inp : str
        Output INP filename.

    emitter_coeff : float, optional
        Emitter coefficient assigned to midpoint junction.
        Default = 0.035.
    """

    wn = wntr.network.WaterNetworkModel(input_inp)

    for pipe_id in list_pipes['Pipe ID']:

        if pipe_id not in wn.pipe_name_list:
            print(f"Pipe '{pipe_id}' not found. Skipping.")
            continue

        pipe = wn.get_link(pipe_id)

        start_node_name = pipe.start_node_name
        end_node_name = pipe.end_node_name

        start_node = wn.get_node(start_node_name)
        end_node = wn.get_node(end_node_name)

        # Coordinates
        x1, y1 = start_node.coordinates
        x2, y2 = end_node.coordinates

        # Midpoint coordinates (add some translation)
        xm = (x1 + x2) / 2.0 + 1.0
        ym = (y1 + y2) / 2.0 + 1.0

        midpoint_node = f"H_{pipe_id}"

        if midpoint_node in wn.node_name_list:
            raise ValueError(
                f"Node '{midpoint_node}' already exists."
            )

        # Elevation at midpoint
        elev1 = getattr(start_node, "elevation", 0.0)
        elev2 = getattr(end_node, "elevation", 0.0)

        midpoint_elev = (elev1 + elev2) / 2.0

        # Create midpoint junction
        wn.add_junction(
            midpoint_node,
            base_demand=0.0,
            demand_pattern=None,
            elevation=midpoint_elev,
            coordinates=(xm, ym)
        )
        
        # Direction vector of original pipe
        dx = x2 - x1
        dy = y2 - y1
        
        pipe_length_xy = (dx**2 + dy**2) ** 0.5
        
        if pipe_length_xy == 0:
            raise ValueError(
                f"Pipe {pipe_id} has coincident start/end coordinates."
            )
        
        # Unit perpendicular vector
        px = -dy / pipe_length_xy
        py = dx / pipe_length_xy
        
        # Emitter node located 1 m from midpoint,
        # perpendicular to original pipe
        xe = xm + px
        ye = ym + py

        emitter_node = f"E_{pipe_id}"
        wn.add_junction(
            emitter_node,
            base_demand=0.0,
            demand_pattern=None,
            elevation=midpoint_elev,
            coordinates=(xe, ye)
        )

        # Assign emitter
        emitterpoint = wn.get_node(emitter_node)
        cvalue = 0.001*float(pipes_interest.loc[pipes_interest['Pipe ID'] == pipe_id]['Coefficient'].values[0])
        emitterpoint.emitter_coefficient = cvalue 

        # Original pipe properties
        length = pipe.length
        diameter = pipe.diameter
        roughness = pipe.roughness
        minor_loss = pipe.minor_loss
        status = pipe.initial_status

        # Close original pipe
        pipe.initial_status = wntr.network.LinkStatus.Closed

        # New pipe IDs
        pipe_a = f"{pipe_id}_A"
        pipe_b = f"{pipe_id}_B"

        # First half pipe
        wn.add_pipe(
            pipe_a,
            start_node_name,
            midpoint_node,
            length=length / 2.0,
            diameter=diameter,
            roughness=roughness,
            minor_loss=minor_loss,
            initial_status=status
        )

        # Second half pipe
        wn.add_pipe(
            pipe_b,
            end_node_name,
            midpoint_node,
            length=length / 2.0,
            diameter=diameter,
            roughness=roughness,
            minor_loss=minor_loss,
            initial_status=status
        )

        
        # Create valve branch

        branch_pipe = f"{pipe_id}_CV"

        wn.add_pipe(
            branch_pipe,
            midpoint_node,
            emitter_node,
            length=0.1,
            diameter=1.0,
            roughness=100,
            check_valve=True,
        )

        print(
            f"Processed {pipe_id}: "
            f"pipe CLOSED, "
            f"add junc {midpoint_node}, "
            f"add pipes {pipe_a} and {pipe_b}, "
            f"add Emitter {emitter_node}, "
            f"add pipe CV {branch_pipe}"
        )

    wntr.network.write_inpfile(wn, output_inp)

    print(f"Saved modified network to {output_inp}")

In [98]:
# load the wdn as INP file
input_inp = 'BBM-EPS.inp'

# which damage scenario to analyse
ds_sel = 'DS5' 

# load the information required 
file_rep = 'BPDRR_reparations.xlsx'

file_distance = 'BBM_distances_'+ds_sel+'.csv'
output_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'
FlagSave = 1
FlagPlot = 1

pipes_interest = pd.read_excel(file_rep, 
                               sheet_name=ds_sel,
                               converters={'Junction':str,'Coefficient':float,'Pipe ID':str})
pipes_interest

,Junction,Coefficient,Pipe ID
0,E3922,2.42800,3922
1,E3398,2.42800,3398
2,E3825,2.42800,3825
3,E4117,2.42800,4117
4,E1902,1.36575,1902
...,...,...,...
103,E587,0.29025,587
104,E5893,0.29025,5893
105,E772,0.29025,772
106,E90,0.29025,90


In [99]:
pipes_interest.dtypes

Junction        object
Coefficient    float64
Pipe ID         object
dtype: object

In [100]:
#np.any(pipes_interest['Pipe ID'] == )
float(pipes_interest['Coefficient'].loc[pipes_interest['Pipe ID']=='4022'].values[0])

0.29025

In [101]:
split_pipes_with_emitter(input_inp,pipes_interest,output_inp)

Processed 3922: pipe CLOSED, add junc H_3922, add pipes 3922_A and 3922_B, add Emitter E_3922, add pipe CV 3922_CV
Processed 3398: pipe CLOSED, add junc H_3398, add pipes 3398_A and 3398_B, add Emitter E_3398, add pipe CV 3398_CV
Processed 3825: pipe CLOSED, add junc H_3825, add pipes 3825_A and 3825_B, add Emitter E_3825, add pipe CV 3825_CV
Processed 4117: pipe CLOSED, add junc H_4117, add pipes 4117_A and 4117_B, add Emitter E_4117, add pipe CV 4117_CV
Processed 1902: pipe CLOSED, add junc H_1902, add pipes 1902_A and 1902_B, add Emitter E_1902, add pipe CV 1902_CV
Processed 3019: pipe CLOSED, add junc H_3019, add pipes 3019_A and 3019_B, add Emitter E_3019, add pipe CV 3019_CV
Processed 91: pipe CLOSED, add junc H_91, add pipes 91_A and 91_B, add Emitter E_91, add pipe CV 91_CV
Processed 1540: pipe CLOSED, add junc H_1540, add pipes 1540_A and 1540_B, add Emitter E_1540, add pipe CV 1540_CV
Processed 2050: pipe CLOSED, add junc H_2050, add pipes 2050_A and 2050_B, add Emitter E_205